# 01 — Extract Channel Taxonomy

Two-phase pipeline that samples ~88 simulators, extracts free-form channel
descriptions (Phase 2), then consolidates them into a two-level taxonomy (Phase 3).

| Phase | What happens | Output |
|-------|-------------|--------|
| **2** | Batched free-form extraction on a stratified sample | `allen_raw_descriptions.json` |
| **3** | One LLM call consolidates all descriptions into a taxonomy | `allen_taxonomy.json` |

Both phases are **resumable**: re-running a cell skips work when the output file
already exists.  Delete the file to force a re-run of that phase.

**Base HH channels (NOT classified):** fast Na⁺ (m³h), delayed-rectifier K⁺ (n⁴), ohmic Leak.

In [1]:
import ast
import json
import os
import re
import subprocess
from pathlib import Path

import dspy
import pandas as pd

**Before running:** set `RESULTS_ROOT` below to the raw experiment output directory
that contains the generated model for *every* particle
(`<timestamp>/<seed>/iter-<i>_p<j>/simulator.py`) — not the curated `main_results/`
folder, which only keeps `best_particle/`/`baseline/` for the best run per seed.

If you only need the already-computed results, you don't need to run this notebook
at all — see `main_results/allen_posterior_mass_analysis/` (`allen_taxonomy.json`,
`allen_raw_descriptions.json`, `allen_classified.csv`), which is what
`figures/Fig_5_posterior_mass_analysis/` and `figures/Fig_F2_cross_seed_rank_stability/`
read from.

In [ ]:
# ── Paths (relative to this notebook's directory) ─────────────────────────────
REPO_ROOT           = Path("../../").resolve()
SUMMARY_CSV         = REPO_ROOT / "main_results/allen/modelsmc/sonnet/summary.csv"
RESULTS_ROOT        = REPO_ROOT / "results/allen_modelsmc_sonnet"
BASE_SIMULATOR_PATH = REPO_ROOT / "modelsmc/tasks/allen/level0/base_simulator.py"

RAW_DESC_JSON = Path("allen_raw_descriptions.json")  # Phase 2 output
TAXONOMY_JSON = Path("allen_taxonomy.json")           # Phase 3 output

# ── Tunable constants ──────────────────────────────────────────────────────────
EXTRACTION_BATCH_SIZE = 8   # simulators per LLM call in Phase 2
SAMPLE_SIZE_FRAC = 0.15 # Fraction of total simulations to sample for Phase 2; set to 1.0 to use all.

# ── Build simulator index: (seed, iteration, particle_index) -> Path ──────────
# Scans ALL timestamp subdirectories; later timestamp wins on duplicates.
sim_index = {}
timestamp_dirs = sorted(
    [d for d in RESULTS_ROOT.iterdir() if d.is_dir() and not d.name.startswith(".")]
)
for ts_dir in timestamp_dirs:
    for seed_dir in ts_dir.iterdir():
        if not seed_dir.is_dir():
            continue
        try:
            seed_val = int(seed_dir.name)
        except ValueError:
            continue
        for particle_dir in seed_dir.iterdir():
            if not particle_dir.is_dir():
                continue
            name = particle_dir.name  # e.g. iter-3_p2
            m = __import__("re").match(r"iter-(\d+)_p(\d+)", name)
            if not m:
                continue
            key = (seed_val, int(m.group(1)), int(m.group(2)))
            sim_index[key] = particle_dir / "simulator.py"
print(f"sim_index built: {len(sim_index)} entries from {len(timestamp_dirs)} timestamp dir(s)")

In [3]:
# ── API key via dotenvx ───────────────────────────────────────────────────────
result = subprocess.run(
    ["dotenvx", "get", "ANTHROPIC_API_KEY"],
    capture_output=True, text=True, cwd=REPO_ROOT,
)
api_key = result.stdout.strip()
if not api_key:
    raise RuntimeError("Could not retrieve ANTHROPIC_API_KEY via dotenvx")
os.environ["ANTHROPIC_API_KEY"] = api_key
print("API key loaded:", api_key[:8] + "...")

lm = dspy.LM(
    model="anthropic/claude-sonnet-4-6",
    api_key=api_key,
    temperature=0.0,
    max_tokens=20_000,
)
dspy.configure(lm=lm)
print("DSPy LM configured:", lm.model)

API key loaded: sk-ant-a...
DSPy LM configured: anthropic/claude-sonnet-4-6


In [4]:
# ── Load summary.csv and build simulator paths ────────────────────────────────
df = pd.read_csv(SUMMARY_CSV)

def get_seed(cfg_str):
    return ast.literal_eval(cfg_str)["seed"]

df["seed"] = df["config"].apply(get_seed)

def sim_path(row):
    key = (int(row["seed"]), int(row["iteration"]), int(row["particle_index"]))
    return sim_index.get(key)  # None if not found

df["sim_path"]   = df.apply(sim_path, axis=1)
df["sim_exists"] = df["sim_path"].apply(lambda p: p is not None and p.exists())

df_usable = (
    df[(df["log_weight"] != float("-inf")) & df["sim_exists"]]
    .copy()
    .reset_index(drop=True)
)

SAMPLE_SIZE = max(1, round(len(df_usable) * SAMPLE_SIZE_FRAC))  # ~15% of usable particles

print(f"Total rows:       {len(df)}")
print(f"Usable particles: {len(df_usable)}")
print(f"Sample size:      {SAMPLE_SIZE} (~15% of usable)")
print(f"log_weight range: [{df_usable['log_weight'].min():.1f}, {df_usable['log_weight'].max():.1f}]")
print(f"Seeds:            {sorted(df_usable['seed'].unique())}")

Total rows:       1511
Usable particles: 1445
Sample size:      217 (~15% of usable)
log_weight range: [-2239613952.0, -232.6]
Seeds:            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [5]:
# ── Stratified sample: 1/4 particles per quality quartile ─────────────────────
df_usable["quality_quartile"] = pd.qcut(
    df_usable["log_weight"], q=4, labels=["Q1_best", "Q2", "Q3", "Q4_worst"]
)
per_stratum = SAMPLE_SIZE // 4
parts = []
for _, grp in df_usable.groupby("quality_quartile", observed=True):
    parts.append(grp.sample(n=min(per_stratum, len(grp)), random_state=42))

sample_df = pd.concat(parts).drop_duplicates("particle_id").reset_index(drop=True)
print(f"Stratified sample: {len(sample_df)} particles")
print(sample_df["quality_quartile"].value_counts().to_string())

base_code = BASE_SIMULATOR_PATH.read_text()
print(f"\nBase simulator loaded ({len(base_code)} chars)")


# ── Helper: parse_json_response ───────────────────────────────────────────────
def parse_json_response(raw: str, fallback=None):
    """Robust JSON parser: strips markdown fences, unwraps dict wrappers."""
    text = raw.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:])
    if text.endswith("```"):
        text = "\n".join(text.split("\n")[:-1])
    text = text.strip()
    parsed = None
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"(\[.*\]|\{.*\})", text, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group())
            except json.JSONDecodeError:
                pass
    if parsed is None:
        if fallback is not None:
            print(f"  WARNING: JSON parse failed. Raw: {text[:200]}")
            return fallback
        raise ValueError(f"Could not parse JSON: {text[:500]}")
    if isinstance(parsed, dict):
        for key in ("results", "descriptions", "classifications", "items", "data"):
            if key in parsed and isinstance(parsed[key], list):
                return parsed[key]
    return parsed


BASE_CHANNEL_DESCRIPTION = (
    "The BASE Hodgkin-Huxley model has EXACTLY THREE ionic currents:\n"
    "  1. Fast Na+ current: m**3 * h gating (alpha_m, beta_m, alpha_h, beta_h), E_Na = +53 mV\n"
    "  2. Delayed-rectifier K+ current: n**4 gating (alpha_n, beta_n), E_K = -107 mV\n"
    "  3. Ohmic Leak current: g_leak * (V - E_leak)\n"
    "Do NOT describe or classify these three base channels. Only describe what was ADDED."
)

Stratified sample: 216 particles
quality_quartile
Q1_best     54
Q2          54
Q3          54
Q4_worst    54

Base simulator loaded (7313 chars)


---
## Phase 2 — Batched Free-Form Extraction

Sends batches of 8 simulators to the LLM alongside the base HH simulator as reference.
The LLM identifies and describes **only what was added beyond Na/K/Leak** in free-form text.
No taxonomy is imposed — this is the bottom-up discovery pass.

**Skipped if `allen_raw_descriptions.json` already exists.** Delete it to re-run.

In [6]:
class ExtractChannelDescriptions(dspy.Signature):
    """You are an expert computational neuroscientist.

    The BASE Hodgkin-Huxley model has EXACTLY THREE ionic currents:
      1. Fast Na+ current: m**3 * h gating, E_Na = +53 mV
      2. Delayed-rectifier K+ current: n**4 gating, E_K = -107 mV
      3. Ohmic Leak current: g_leak * (V - E_leak)

    You are given the base simulator code and a batch of modified simulators.
    For EACH snippet, identify and describe ONLY what was ADDED beyond the base.

    Look for:
    - New gating variables (e.g. q, r, s, p, a, b) with their own kinetic functions
    - New terms in the tau_V_inv and V_inf (or V_num) blocks
    - New state variable update equations in the simulation loop
    - Comments or docstrings that name the channel type

    For each snippet, fill these STRUCTURED FIELDS carefully:
    - n_channels_added: integer (0 = base only, 1, or 2 for two new channels)
    - channel_family: K_channel, Na_channel, Ca_channel, HCN, mixed, or unknown
    - activation_power: the exponent of the activation gate in the open-probability formula.
      Examples: "p^1" (linear), "p^2", "p^3", "a^1", "instantaneous" (no dynamic variable),
      "p^1*q^1" if both activation and inactivation are linear, etc. Write "none" if base only.
    - has_inactivation_gate: true if there is a separate inactivation variable (e.g. q, b, h_new);
      false if only an activation variable or instantaneous.
    - tau_type: how the time constant is implemented:
        "voltage_dependent" — tau varies with V (bell-shaped, Boltzmann-derived, or alpha/beta)
        "fixed_scalar"      — tau is a learned scalar (e.g. directly from param_j, clamped constant)
        "log_scaled"        — tau = exp(param) or log-mapping is used
        "instantaneous"     — no dynamic state variable at all (algebraic activation)
        "unknown"           — cannot determine from the code
    - description: free-form text — ion type, inactivating or not, time scale, reversal
      potential, gating variable name(s), any distinguishing features.

    Return a JSON array with one entry per snippet IN ORDER:
    [
      {
        "snippet_num": 1,
        "n_channels_added": 1,
        "channel_family": "K_channel",
        "activation_power": "p^1",
        "has_inactivation_gate": false,
        "tau_type": "fixed_scalar",
        "description": "Slow non-inactivating K+ M-type current. Single gate p, sigmoid p_inf, tau_p from param_j (fixed scalar). Reversal E_K."
      },
      ...
    ]
    If snippet adds NOTHING beyond base HH: n_channels_added=0, all structural fields="none"/"false".
    If TWO channels added: describe both, set n_channels_added=2, fill fields for the PRIMARY channel.
    """
    base_simulator:    str = dspy.InputField(desc="Base HH simulator code (reference)")
    batch_snippets:    str = dspy.InputField(desc="Snippets separated by ### SNIPPET N ### headers")
    descriptions_json: str = dspy.OutputField(desc="JSON array, one object per snippet in order")


if RAW_DESC_JSON.exists():
    print(f"Loading existing descriptions from {RAW_DESC_JSON}")
    with open(RAW_DESC_JSON) as f:
        all_descriptions = json.load(f)
    print(f"Loaded {len(all_descriptions)} descriptions")
else:
    extract      = dspy.Predict(ExtractChannelDescriptions)

    all_descriptions = []
    n_batches = (len(sample_df) + EXTRACTION_BATCH_SIZE - 1) // EXTRACTION_BATCH_SIZE

    for batch_idx in range(n_batches):
        start     = batch_idx * EXTRACTION_BATCH_SIZE
        batch     = sample_df.iloc[start : start + EXTRACTION_BATCH_SIZE]
        batch_ids = batch["particle_id"].tolist()

        snippets_text = ""
        for i, (_, row) in enumerate(batch.iterrows(), 1):
            code_text     = Path(row["sim_path"]).read_text()
            snippets_text += f"### SNIPPET {i} ###\n{code_text}\n\n"

        try:
            res    = extract(base_simulator=base_code, batch_snippets=snippets_text)
            parsed = parse_json_response(res.descriptions_json, fallback=[])
        except Exception as e:
            print(f"  Batch {batch_idx+1}/{n_batches}: ERROR — {e}")
            parsed = []

        for j in range(len(batch)):
            item = dict(parsed[j]) if j < len(parsed) else {
                "snippet_num": j + 1, "n_channels_added": -1, "description": "MISSING"
            }
            item["particle_id"] = batch_ids[j]
            item["log_weight"]  = float(batch.iloc[j]["log_weight"])
            all_descriptions.append(item)

        done = min(start + EXTRACTION_BATCH_SIZE, len(sample_df))
        print(f"  Batch {batch_idx+1}/{n_batches}: {len(parsed)} parsed ({done}/{len(sample_df)} particles)")

    with open(RAW_DESC_JSON, "w") as f:
        json.dump(all_descriptions, f, indent=2)
    print(f"\nSaved {len(all_descriptions)} descriptions -> {RAW_DESC_JSON}")

print("\nSample descriptions:")
for d in all_descriptions[:4]:
    pid = str(d.get("particle_id", "?"))[:8]
    act = d.get("activation_power", "?")
    tau = d.get("tau_type", "?")
    inact = d.get("has_inactivation_gate", "?")
    print(f"  [{pid}...] n_added={d.get('n_channels_added','?')} | act={act} | tau={tau} | inact={inact}")
    print(f"    {str(d.get('description',''))[:110]}")

  Batch 1/27: 8 parsed (8/216 particles)
  Batch 2/27: 8 parsed (16/216 particles)
  Batch 3/27: 8 parsed (24/216 particles)
  Batch 4/27: 8 parsed (32/216 particles)
  Batch 5/27: 8 parsed (40/216 particles)
  Batch 6/27: 8 parsed (48/216 particles)
  Batch 7/27: 8 parsed (56/216 particles)
  Batch 8/27: 8 parsed (64/216 particles)
  Batch 9/27: 8 parsed (72/216 particles)
  Batch 10/27: 8 parsed (80/216 particles)
  Batch 11/27: 8 parsed (88/216 particles)
  Batch 12/27: 8 parsed (96/216 particles)
  Batch 13/27: 8 parsed (104/216 particles)
  Batch 14/27: 8 parsed (112/216 particles)
  Batch 15/27: 8 parsed (120/216 particles)
  Batch 16/27: 8 parsed (128/216 particles)
  Batch 17/27: 8 parsed (136/216 particles)
  Batch 18/27: 8 parsed (144/216 particles)
  Batch 19/27: 8 parsed (152/216 particles)
  Batch 20/27: 8 parsed (160/216 particles)
  Batch 21/27: 8 parsed (168/216 particles)
  Batch 22/27: 8 parsed (176/216 particles)
  Batch 23/27: 8 parsed (184/216 particles)
  Batch 24

---
## Phase 3 — Taxonomy Consolidation

One LLM call takes all free-form descriptions and produces a structured two-level
taxonomy (family → subtype). The number of subtypes is not fixed; the LLM decides
based on the semantic diversity it finds.

Grouping philosophy:
- Same biophysical mechanism + same ion → same subtype (even with different parameter values)
- Different gating structure (inactivating vs non-inactivating) or different ion → different subtypes
- Two added channels → `multi_{family1}+{family2}` (families sorted alphabetically)

**Skipped if `allen_taxonomy.json` already exists.** Delete it to re-run.

In [8]:
class ConsolidateTaxonomy(dspy.Signature):
    """You are an expert computational neuroscientist building a taxonomy of
    additional ion channels discovered in Hodgkin-Huxley neuron models.

    The BASE HH model has EXACTLY THREE ionic currents (NOT to be classified):
      1. Fast Na+ current: m**3 * h gating, E_Na = +53 mV
      2. Delayed-rectifier K+ current: n**4 gating, E_K = -107 mV
      3. Ohmic Leak current: g_leak * (V - E_leak)

    You receive a JSON list of structured descriptions of channels ADDED beyond the base.
    Each entry has: channel_family, activation_power, has_inactivation_gate, tau_type, description.
    Build a two-level taxonomy (family -> subtype).

    TAXONOMY SCHEMA:
    - `family_id` IS the canonical channel-type name (e.g. I_M, I_NaP, I_A, I_h).
      Do NOT use broad categories like "K_channel" or "Na_channel" as family_id.
    - `subtype_id` is a specific functional variant: format `{family_id}_{descriptor}`.
    - For base_only and unknown: subtype_id == family_id (no descriptor needed).
    - Multi-channel: family_id = "multi_K+Na", subtype_id = "multi_K+Na_{descriptor}".

    SPLITTING RULES:
    1. For every family with 3+ members in the input, you MUST produce AT LEAST 2 subtypes.
       A single-subtype family is only acceptable when fewer than 3 members exist.
    2. Use the structured fields (activation_power, has_inactivation_gate, tau_type) to find
       the most meaningful split dimension. You decide which field is most informative per family.
       More than 2 subtypes are fine if the data supports it.
    3. Split on STRUCTURAL CODE DIFFERENCES, not parameter values.
    4. Always include base_only and unknown entries.

    Return a JSON object:
    {
      "taxonomy": [
        {
          "family_id": "I_M",
          "subtype_id": "I_M_voltage_dep_tau",
          "name": "...",
          "description": "...",
          "split_criterion": "tau_type=voltage_dependent",
          "is_multi": false,
          "example_phrases": ["bell-shaped tau", "voltage-dependent tau_p"]
        },
        ...
      ],
      "base_only_subtype_id": "base_only",
      "unknown_subtype_id": "unknown"
    }
    """
    base_channel_info:     str = dspy.InputField(desc="Description of the three base channels (NOT to appear in taxonomy)")
    raw_descriptions_json: str = dspy.InputField(desc="JSON list of structured descriptions from Phase 2 (with activation_power, tau_type, etc.)")
    taxonomy_json:         str = dspy.OutputField(desc="JSON taxonomy object as specified")


if TAXONOMY_JSON.exists():
    print(f"Taxonomy already exists — loading from {TAXONOMY_JSON}")
    with open(TAXONOMY_JSON) as f:
        taxonomy = json.load(f)
else:
    consolidate = dspy.Predict(ConsolidateTaxonomy)
    compact = [
        {
            "n_channels_added":      d.get("n_channels_added"),
            "channel_family":        d.get("channel_family"),
            "activation_power":      d.get("activation_power", "unknown"),
            "has_inactivation_gate": d.get("has_inactivation_gate", "unknown"),
            "tau_type":              d.get("tau_type", "unknown"),
            "description":           d.get("description", ""),
        }
        for d in all_descriptions
        if d.get("n_channels_added", -1) >= 0
    ]
    desc_str = json.dumps(compact, indent=2)
    print(f"Consolidating {len(compact)} descriptions (~{len(desc_str)//1000}k chars) ...")

    from collections import Counter
    fam_counts = Counter(d.get("channel_family") for d in compact if d.get("n_channels_added", 0) > 0)
    print("Family member counts in sample:", dict(fam_counts))

    res      = consolidate(base_channel_info=BASE_CHANNEL_DESCRIPTION, raw_descriptions_json=desc_str)
    taxonomy = parse_json_response(res.taxonomy_json)

    with open(TAXONOMY_JSON, "w") as f:
        json.dump(taxonomy, f, indent=2)
    print(f"Taxonomy saved -> {TAXONOMY_JSON}")

# ── Display taxonomy ──────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DISCOVERED TAXONOMY")
print("=" * 70)
print(f"base_only label : '{taxonomy.get('base_only_subtype_id', 'base_only')}'")
print(f"unknown label   : '{taxonomy.get('unknown_subtype_id', 'unknown')}'")
print(f"\n{len(taxonomy['taxonomy'])} channel subtypes:\n")

by_family = {}
for ct in taxonomy["taxonomy"]:
    by_family.setdefault(ct.get("family_id", "other"), []).append(ct)

for fam in sorted(by_family):
    print(f"  [{fam}] ({len(by_family[fam])} subtype(s))")
    for st in by_family[fam]:
        multi_tag = "  [MULTI]" if st.get("is_multi") else ""
        split_crit = st.get("split_criterion", "")
        print(f"    {st['subtype_id']:36s} split: {split_crit}{multi_tag}")
        print(f"      {st.get('description', '')[:110]}")
        phrases = ", ".join(st.get("example_phrases", [])[:4])
        if phrases:
            print(f"      keywords: {phrases}")
    print()
print("=" * 70)

Taxonomy already exists — loading from allen_taxonomy.json

DISCOVERED TAXONOMY
base_only label : 'base_only'
unknown label   : 'unknown'

11 channel subtypes:

  [I_M] (2 subtype(s))
    I_M_fixed_tau                        split: tau_type=fixed_scalar
      Slow non-inactivating M-type K+ current (Kv7/KCNQ) with a single activation gate (p or w), Boltzmann steady-st
      keywords: fixed scalar tau_p from params, voltage-independent time constant, tau_M = clamp(params[:,9], min=1.0), precomputed decay factor
    I_M_voltage_dep_tau                  split: tau_type=voltage_dependent
      Slow non-inactivating M-type K+ current (Kv7/KCNQ) with a single activation gate (p or w), Boltzmann or alpha/
      keywords: bell-shaped tau_p, voltage-dependent tau via alpha/beta, tau_max/(3.3*exp(dv/20)+exp(-dv/20)), Wang 1998 formulation

  [I_M_plus_I_A] (1 subtype(s))
    I_M_plus_I_A                         split: has_inactivation_gate=true AND secondary=I_A  [MULTI]
      Two-channel combin

---
## ⚠️ STOP — Review the Taxonomy Before Proceeding

Inspect the taxonomy printed above. Options:

- **Edit `allen_taxonomy.json` directly** to merge subtypes that are too similar,
  split ones that capture two distinct mechanisms, or rename entries.
- **Delete `allen_taxonomy.json`** and re-run the Phase 3 cell to get a fresh LLM attempt.
- **Delete both JSON files** to restart from scratch (different sample or prompt).

When satisfied with the taxonomy, open **`02_classify_particles.ipynb`**.